# PROMPT/02 · 비교 가능한 프롬프트 변경

고객 문의 분류를 예로 출력 명세, 소수 예시, 형식·의미 검사, 변경 근거를 연결합니다. 필수 셀은 인터넷·모델·API 키 없이 실행됩니다. Python 3.11.14 공통 `.venv` 커널을 선택하세요.

**demo는 고정 응답 재생입니다. 실제 모델 추론이 아니며 프롬프트 편집으로 demo 응답은 바뀌지 않습니다.** 기대되는 개선 수치는 교육용 예시이며 모델의 성능 증거가 아닙니다.

In [ ]:
import json
import sys
from pathlib import Path

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "week02/lab/prompt02").is_dir())
LAB = REPO / "week02/lab"
if str(LAB) not in sys.path:
    sys.path.insert(0, str(LAB))
assert sys.version_info[:2] == (3, 11), "공통 Python 3.11 커널을 선택하세요."
# 노트북 실행 위치와 무관하게 import하도록 위에서 모듈 경로를 먼저 설정합니다.
from prompt02.contracts import ExperimentRequest, TicketResult  # noqa: E402
from prompt02.core import fingerprint, load_dataset, render_prompt, validate_output  # noqa: E402
from prompt02.experiments import run_experiment  # noqa: E402

print("Python", sys.version.split()[0], "| PROMPT/02 import OK")

## 1. 업무와 평가 기준
가상 문의 8건입니다. 정답 라벨은 실습에서 정한 업무 규칙이며 실제 기업의 정책이 아닙니다. `rationale`을 읽고 T07의 교환 우선 분류와 검토 기준을 설명해보세요. 프롬프트의 소수 예시 3건은 이 평가 문의와 별개입니다.

In [ ]:
import pandas as pd

dataset = load_dataset()
cases = dataset["cases"]
by_id = {case["id"]: case for case in cases}
pd.DataFrame(cases)[["id", "tag", "text", "expected_category", "expected_review", "rationale"]]

## 2. 동일한 문의, 세 개의 프롬프트
v1은 간단한 업무 지시, v2는 출력 명세와 업무 규칙, v3는 v2 규칙에 경계 예시를 추가합니다. 메시지의 `system`에는 지시를, `user`에는 문의 데이터를 넣습니다. 경계를 나누는 것만으로 모든 지시문 주입이 차단되지는 않습니다.

In [ ]:
preview = {version: render_prompt(version, by_id["T03"]) for version in ["v1", "v2", "v3"]}
assert len({item["input_hash"] for item in preview.values()}) == 1
assert len({item["prompt_hash"] for item in preview.values()}) == 3
print(preview["v3"]["template"])
print("사용자 메시지:", preview["v3"]["messages"][1]["content"])
print("출력 명세:", json.dumps(TicketResult.model_json_schema(), ensure_ascii=False, indent=2))

## 3. 오프라인 비교 실행
같은 문의 8건을 v1/v2/v3에 전달한 24개 결과를 비교합니다. 모델을 호출하지 않고 `data/demo_responses.json`을 재생합니다. 전체 통과 0/8, 4/8, 8/8은 이 고정 사례의 결과이며 프롬프트 개선 효과를 측정한 값이 아닙니다.

In [ ]:
report = await run_experiment(ExperimentRequest(provider="demo"))
assert report["simulation"] is True
assert len(report["rows"]) == 24
summary = pd.DataFrame(
    [
        {
            "version": version,
            **{
                metric: f"{value[metric]['passed']}/{value['total']}"
                for metric in ["format_ok", "routing_ok", "promise_ok", "review_ok", "overall_ok"]
            },
        }
        for version, value in report["summary"].items()
    ]
)
summary

## 4. 실패를 구분하는 기준
`format_ok`: 엄격한 JSON 명세. `routing_ok`: 분류. `promise_ok`: 근거 없는 확정 약속을 탐지하지 않았는지. `review_ok`: 담당자 검토와 우선순위. 형식 실패 시 의미 검사는 `None`(미평가)입니다. 확정 약속의 정규표현식 검사는 오탐·미탐 가능성이 있습니다.

In [ ]:
for case_id, version in [("T01", "v1"), ("T03", "v2"), ("T08", "v2")]:
    row = next(r for r in report["rows"] if r["case_id"] == case_id and r["version"] == version)
    print(case_id, version, "입력:", row["input"], "출력:", row["raw_output"])
    print("기대 기준:", row["expected"])
    print("검사:", row["validation"])

In [ ]:
# 모델 출력은 코드가 아니라 검사할 데이터입니다. eval/exec를 사용하지 않습니다.
malformed = validate_output("```json\n{}\n```", by_id["T01"])
assert malformed["format_ok"] is False and malformed["routing_ok"] is None
wrong_but_valid = json.dumps(
    {
        "category": "refund",
        "priority": "normal",
        "needs_review": False,
        "reply": "오늘 전액 환불해 드리겠습니다.",
    },
    ensure_ascii=False,
)
checked = validate_output(wrong_but_valid, by_id["T01"])
assert checked["format_ok"] is True
assert checked["routing_ok"] is False and checked["promise_ok"] is False
checked

## 5. 바뀐 것의 기록
SHA-256 해시는 원문이 달라졌는지 식별하는 값이지 품질 점수가 아닙니다. 프롬프트만 바꾸면 입력 해시는 같고 프롬프트 해시는 달라져야 합니다. 아래 변경은 메모리에서만 하며 원본 파일은 수정하지 않습니다.

In [ ]:
original = preview["v3"]
changed = render_prompt(
    "v3", by_id["T03"], template=original["template"] + "\n답변을 80자 이내로 작성하세요."
)
assert original["input_hash"] == changed["input_hash"]
assert original["prompt_hash"] != changed["prompt_hash"]
assert fingerprint(dataset) == report["dataset_hash"]
print("원본 프롬프트:", original["prompt_hash"])
print("변경 프롬프트:", changed["prompt_hash"])
print("고정 입력:", original["input_hash"])

## 6. 다음 검증 후보와 보고서
선택 근거에 개선된 사례, 남은 한계, 다음 평가 계획을 작성합니다. 실제 배포는 수행하지 않습니다. 아래 보고서는 합성 문의와 교육용 결정만 담고 `week02/lab/reports/`에 저장됩니다. 본인·타인의 개인정보는 작성하지 마세요.

In [ ]:
from fastapi.testclient import TestClient
from prompt02.main import app

with TestClient(app) as client:
    response = client.post(
        "/api/experiments", json={"provider": "demo", "case_ids": ["T01", "T03", "T06"]}
    )
    response.raise_for_status()
    api_report = response.json()
    decision = client.post(
        "/api/decisions",
        json={
            "run_id": api_report["run_id"],
            "version": "v3",
            "reason": "형식과 경계 사례를 확인했다. "
            "고정 응답이므로 실제 모델과 별도 평가 세트에서 다시 검증한다.",
        },
    )
    decision.raise_for_status()
    assert decision.json()["deployed"] is False
    exported = client.get(f"/api/experiments/{api_report['run_id']}/export").json()

reports_dir = LAB / "reports"
reports_dir.mkdir(exist_ok=True)
output_path = reports_dir / f"prompt02-{exported['run_id']}.json"
with output_path.open("x", encoding="utf-8") as handle:
    json.dump(exported, handle, ensure_ascii=False, indent=2)
print("저장:", output_path.relative_to(REPO))
print("실험 내 선택 기록:", exported["decisions"][0]["reason"])

## 7. 선택 · 실제 로컬 모델
이 셀은 기본적으로 모델을 호출하지 않습니다. Ollama와 `qwen3:4b-instruct`를 준비한 뒤 직접 `True`로 바꾼 경우에만 T01을 실제 호출합니다. 기본 설정: `think:false`, `temperature:0`, `seed:42`, `num_ctx:4096`, `num_predict:384`, 개별 호출 최대 35초. 오류가 나면 나머지 호출은 미실행이며 demo로 자동 전환하지 않습니다.

In [ ]:
RUN_REAL_OLLAMA = False
if RUN_REAL_OLLAMA:
    real_report = await run_experiment(ExperimentRequest(provider="ollama", case_ids=["T01"]))
    print(json.dumps(real_report["summary"], ensure_ascii=False, indent=2))
    for row in real_report["rows"]:
        print(row["version"], row["error"] or row["raw_output"])
else:
    print("선택 Ollama 실행 생략 · 필수 오프라인 실습 완료")

## 제출 전 설명 연습

1. JSON 형식이 맞는데도 T03의 v2가 실패한 이유는 무엇인가요?
2. T08의 환불 답변에서 확정할 수 없는 정보는 무엇인가요?
3. 소수 예시와 평가 데이터를 분리하는 이유는 무엇인가요?
4. 실제 모델에서 v3가 더 느리거나 성능이 나빠졌다면 어떤 기록으로 비교하겠습니까?
5. 새 문의 3건을 만든다면 어떤 경계 사례를 추가하겠습니까?

브라우저 실습을 실행했다면 보고서를 저장한 뒤 서버 터미널에서 `Ctrl+C`로 종료합니다. 다음 단계는 이 결과를 포트폴리오의 문제 정의, 평가 기준, 개선 근거에 연결하는 것입니다.